In [0]:
# 1. IMPORTS AND SETTINGS
from pyspark.sql import functions as F
from datetime import datetime

# Change only this date when processing the next day's files
process_date = "05_04_2026"

github_base_path = ("/Workspace/Users/ankurkamboj.ynr@gmail.com/""time_n_work_utilization_at_7trents/data_ingest_github/")

target_schema = "severntrent.bronze_dev"

# Source filename -> target table name
fact_sources = {
    "FactWork": "fact_work",
    "FactShift": "fact_shift",
    "FactShrinkage": "fact_shrinkage"
}

print("Processing date:", process_date)

In [0]:
# 2. READ ONE DAILY CSV AND ADD AUDIT COLUMNS

def bronze_ingestor(source_name, process_date):
    file_name = source_name + "_" + process_date + ".csv"
    file_path = github_base_path + process_date + "/" + file_name

    df = (spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path))

    df = df.withColumn("source_file", F.lit(file_name))
    df = df.withColumn("load_date", F.to_date(F.lit(process_date), "dd_MM_yyyy"))
    df = df.withColumn("ingestion_ts", F.current_timestamp())

    load_type = "FULL" if process_date == "01_04_2026" else "DELTA"
    df = df.withColumn("load_type", F.lit(load_type))

    print("Read", file_name, "-", df.count(), "records")
    return df

In [0]:
# 3. WRITE DAILY DATA SAFELY
def daily_incremental_load(source_name, process_date):
    table_name = fact_sources[source_name]
    target_table = target_schema + "." + table_name
    load_date = datetime.strptime(process_date, "%d_%m_%Y").date()

    # Do not load the same day twice
    if spark.catalog.tableExists(target_table):
        already_loaded = (spark.table(target_table)
                          .filter(F.col("load_date") == F.lit(load_date))
                          .limit(1).count() > 0)
        if already_loaded:
            print(table_name, "-", process_date, "already loaded; skipping")
            return
    elif process_date != "01_04_2026":
        raise ValueError("Day 1 FULL must be loaded before a DELTA day: " + target_table)

    df = bronze_ingestor(source_name, process_date)

    # Create table on Day 1; append only for later dates
    if spark.catalog.tableExists(target_table):
        df.write.format("delta").mode("append").saveAsTable(target_table)
    else:
        df.write.format("delta").saveAsTable(target_table)

    print("Saved", table_name, "for", process_date)

In [0]:
# 4. RUN THE DAILY PIPELINE FOR ALL THREE FACT TABLES

#This cell executes the pipeline. A Python loop processes all three fact tables using the same incremental loading function, reducing code duplication.

spark.sql("CREATE SCHEMA IF NOT EXISTS " + target_schema)

for source_name in fact_sources:
    daily_incremental_load(source_name, process_date)

In [0]:
# 5. VALIDATE ALL THREE BRONZE TABLES

#validate the Bronze tables by counting records and grouping them by processing date and load type. This confirms that daily records have been appended as expected.

for table_name in fact_sources.values():
    target_table = target_schema + "." + table_name
    if spark.catalog.tableExists(target_table):
        df = spark.table(target_table)
        print("\nTable:", target_table)
        print("Total records:", df.count())
        df.groupBy("load_date", "load_type").count().orderBy("load_date").show()
    else:
        print("Not yet created:", target_table)